# Data Management & Data Cleaning – `noleggi` (Tutte le sedi)

**Obiettivo:** Eseguire l'intero flusso di *Data Profiling*, *Data Cleaning* e *Data Validation* sui contratti di noleggio di tutte le sedi (**ALB, BAR, CAS, STR, TOR**) utilizzando esclusivamente le librerie standard di Data Management (`pandas`, `numpy`), fino ad ottenere tabelle pulite e coerenti con le anagrafiche (`flotta_con_tariffe.csv`, `tariffe.csv`).

### Fasi del notebook
1. **Caricamento dei dati** delle 5 sedi e delle tabelle anagrafiche
2. **Data Profiling** (dimensioni, tipi, valori mancanti, duplicati, distribuzioni)
3. **Pulizia delle chiavi e integrità referenziale** (`map` / incrocio con `flotta_con_tariffe.csv`)
4. **Standardizzazione delle variabili categoriche, anagrafiche e extra** (`.str`, `.replace()`)
5. **Normalizzazione orari (`HH:MM`), parsing date e coerenza temporale** (`pd.to_datetime()`)
6. **Pulizia variabili numeriche, outlier e ricalcolo totali** (`pd.to_numeric()`, `np.where()`)
7. **Deduplicazione e controllo qualità post-pulizia** (`.drop_duplicates()`, verifica con `_ground_truth`)
8. **Esportazione CSV puliti** (per singola sede e tabella consolidata)

## 1. Import delle librerie e caricamento dei dataset

In [1]:
import os
import numpy as np
import pandas as pd

SEDI = ["ALB", "BAR", "CAS", "STR", "TOR"]
BASE_DATI = "../buby_rent_dati_sporchi"
BASE_FILES = "../../files"
OUT_BASE = "../../dati_puliti"

flotta = pd.read_csv(f"{BASE_FILES}/flotta_con_tariffe.csv")
tariffe = pd.read_csv(f"{BASE_FILES}/tariffe.csv")

# Caricamento e unione dei 5 dataset grezzi mantenendo traccia della sede di origine
df_raw = pd.concat(
    [pd.read_csv(f"{BASE_DATI}/{s}/strutturati/noleggi_{s}.csv").assign(_sede_origine=s) for s in SEDI],
    ignore_index=True
)

print(f"Dataset complessivo caricato: {df_raw.shape[0]} righe, {df_raw.shape[1]} colonne")
df_raw.head()

Dataset complessivo caricato: 5143 righe, 41 colonne


,id_noleggio,data_prenotazione,canale_prenotazione,id_veicolo,targa,tipologia_mezzo,sede_ritiro,sede_riconsegna,data_ritiro,ora_ritiro,...,penale_ritardo_eur,sconto_eur,importo_totale_eur,cauzione_eur,metodo_pagamento,stato_noleggio,danni_segnalati,valutazione_cliente,note_operatore,_sede_origine
0,NOL-ALB-00001,2025-09-16,web,ALB-05,GR153MD,auto,ALB,ALB,2025-11-01,10:16,...,-,0.00,110.50,600.00,bonifico,chiuso ok,S,NaN,NaN,ALB
1,NOL-ALB-00181,27 gen 2026,web,ALB-06,GN630ZK,auto,ALB,CAS,10 feb 2026,17:43,...,NaN,NaN,"478,50","600,00",Carta di credito,concluso,N,4 stelle,NaN,ALB
2,NOL-ALB-00003,22/09/2025,WEB,EB-ALB-003,NaN,e-bike,ALB,Buby Rent Alba,06/11/2025,13:36,...,"0,00",0,13,100,carta debito,concluso,N,5,NaN,ALB
3,NOL-ALB-00004,23/09/2025,app,ALB-07,GP905TB,furgone,Buby Rent Alba,ALB,07/11/2025,15:15,...,0.00,0.00,"93,50","600,00",carta di credito,conclusa,NaN,★★★★,NaN,ALB
4,NOL-ALB-00005,30/09/2025,sito,EB-ALB-002,NaN,E-Bike,ALB,ALB,14/11/2025,16:09,...,0.00,-,17.00,200.00,cash,concluso,no,NaN,NaN,ALB


## 2. Data Exploration & Data Profiling iniziale

In [2]:
# Riepilogo valori mancanti e duplicati prima della pulizia
profiling = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'non_null': df_raw.notna().sum(),
    'null_count': df_raw.isna().sum(),
    'null_pct': (df_raw.isna().mean() * 100).round(2),
    'n_unique': df_raw.nunique()
})
print("Duplicati esatti intera riga:", df_raw.duplicated().sum())
print("Duplicati su id_noleggio grezzo:", df_raw['id_noleggio'].duplicated().sum())
profiling[profiling['null_count'] > 0].sort_values('null_count', ascending=False)

Duplicati esatti intera riga: 88
Duplicati su id_noleggio grezzo: 142


,dtype,non_null,null_count,null_pct,n_unique
note_operatore,str,1365,3778,73.46,28
piva_cliente,str,2098,3045,59.21,350
valutazione_cliente,str,2099,3044,59.19,24
targa,str,3124,2019,39.26,163
km_percorsi,float64,3306,1837,35.72,842
extra_selezionati,str,3628,1515,29.46,111
extra_km_eur,str,3805,1338,26.02,140
penale_ritardo_eur,str,3831,1312,25.51,255
danni_segnalati,str,3965,1178,22.90,11
livello_carb_batt_riconsegna,str,4203,940,18.28,240


In [3]:
# Ispezione delle principali variabili categoriche sporche
for col in ['canale_prenotazione', 'tipologia_mezzo', 'sede_ritiro', 'tipo_tariffa', 'metodo_pagamento', 'stato_noleggio']:
    print(f"--- {col} ---")
    print(df_raw[col].value_counts(dropna=False).to_dict())

--- canale_prenotazione ---
{'web': 1104, 'app': 656, 'sportello': 482, 'telefono': 464, 'portale partner': 330, 'app mobile': 247, 'App': 235, 'sito': 213, 'WEB': 209, 'Sito web': 203, 'online': 184, 'agenzia': 169, 'in sede': 162, 'tel': 102, 'Telefono': 101, 'booking partner': 84, 'chiamata': 79, 'portale': 64, 'Partner': 55}
--- tipologia_mezzo ---
{'auto': 1683, 'e-bike': 1393, 'furgone': 509, 'AUTO': 258, 'autovettura': 257, 'Auto': 248, 'E-Bike': 199, 'ebike': 188, 'bici elettrica': 185, 'Furgone': 79, 'furg.': 76, 'FURGONE': 68}
--- sede_ritiro ---
{'TOR': 1285, 'STR': 680, 'ALB': 639, 'CAS': 609, 'BAR': 502, 'Torino Lingotto': 138, 'Torino': 130, 'Lingotto': 124, 'tor': 105, 'Alba': 78, 'stresa VB': 74, 'str': 72, 'Stresa': 67, 'Casale Monferrato': 62, 'Bardo': 60, 'Buby Rent Casale M.to': 60, 'Buby Rent Alba': 58, 'alb': 56, 'Buby Rent Stresa': 55, 'Casale': 54, 'Buby Rent Bardonecchia': 52, 'alba (CN)': 51, 'cas': 49, 'Bardonecchia': 42, 'bar': 41}
--- tipo_tariffa ---
{'gio

## 3. Pulizia delle chiavi e integrità referenziale (`id_noleggio`, `targa`, `id_veicolo`)

In [4]:
df = df_raw.copy()

# 3.1 Standardizzazione id_noleggio al formato NOL-<SEDE>-00000
num_nol = df['id_noleggio'].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(5)
sede_nol = df['id_noleggio'].astype(str).str.upper().str.extract(r'([A-Z]{3})(?=[^A-Z]*\d)', expand=False).fillna(df['_sede_origine'])
df['id_noleggio'] = 'NOL-' + sede_nol + '-' + num_nol

# 3.2 Pulizia targa (maiuscolo, senza spazi o trattini)
df['targa'] = (
    df['targa']
    .astype('string')
    .str.upper()
    .str.replace(r'[^A-Z0-9]', '', regex=True)
    .replace({'': pd.NA, 'ND': pd.NA, 'NAN': pd.NA})
)

# 3.3 Pulizia formato id_veicolo (gestione targhe finite in id_veicolo, minuscole, spazi, zeri iniziali)
v_clean = df['id_veicolo'].astype('string').str.strip().str.upper().str.replace(r'\s+', '-', regex=True)
is_targa_in_id = v_clean.str.match(r'^[A-Z]{2}\d{3}[A-Z]{2}$', na=False)
df.loc[is_targa_in_id & df['targa'].isna(), 'targa'] = v_clean[is_targa_in_id & df['targa'].isna()]
v_clean = v_clean.where(~is_targa_in_id, pd.NA)

eb_parts = v_clean.str.extract(r'^EB-?([A-Z]{3})-?(\d+)$')
auto_parts = v_clean.str.extract(r'^([A-Z]{3})-?(\d+)$')

df['id_veicolo'] = np.where(
    eb_parts[0].notna(),
    'EB-' + eb_parts[0].fillna('') + '-' + eb_parts[1].fillna('').str.zfill(3),
    np.where(
        auto_parts[0].notna(),
        auto_parts[0].fillna('') + '-' + auto_parts[1].fillna('').str.zfill(2),
        pd.NA
    )
)

# Controllo validità rispetto all'anagrafica flotta_con_tariffe.csv
valid_ids = set(flotta['id_veicolo'])
df.loc[~df['id_veicolo'].isin(valid_ids), 'id_veicolo'] = pd.NA

# Recupero id_veicolo mancante tramite lookup su targa e recupero targhe dall'anagrafica
targa_map = flotta.dropna(subset=['targa']).set_index('targa')['id_veicolo']
id_targa_map = flotta.set_index('id_veicolo')['targa']
df['id_veicolo'] = df['id_veicolo'].fillna(df['targa'].map(targa_map))
df['targa'] = df['targa'].fillna(df['id_veicolo'].map(id_targa_map))

# Il conteggio finale e' stampato dopo il recupero con progressione km (cella 13)

## 4. Standardizzazione variabili categoriche, anagrafica clienti ed extra

In [5]:
# 4.1 Dizionari di mappatura canonica
mappa_sedi = {
    'tor': 'TOR', 'torino': 'TOR', 'torino lingotto': 'TOR', 'lingotto': 'TOR', 'buby rent torino lingotto': 'TOR',
    'alb': 'ALB', 'alba': 'ALB', 'alba (cn)': 'ALB', 'buby rent alba': 'ALB',
    'bar': 'BAR', 'bardonecchia': 'BAR', 'bardo': 'BAR', 'buby rent bardonecchia': 'BAR',
    'str': 'STR', 'stresa': 'STR', 'stresa vb': 'STR', 'buby rent stresa': 'STR',
    'cas': 'CAS', 'casale': 'CAS', 'casale monferrato': 'CAS', 'buby rent casale m.to': 'CAS', 'buby rent casale monferrato': 'CAS'
}
mappa_canale = {
    'web': 'web', 'sito': 'web', 'sito web': 'web', 'online': 'web',
    'app': 'app', 'app mobile': 'app',
    'sportello': 'sportello', 'in sede': 'sportello', 'agenzia': 'sportello',
    'telefono': 'telefono', 'tel': 'telefono', 'chiamata': 'telefono',
    'portale partner': 'portale partner', 'booking partner': 'portale partner', 'portale': 'portale partner', 'partner': 'portale partner'
}
# Valori allineati a TIPI_VEICOLO dell'app (buby-rent/api/app/models.py)
mappa_tipologia = {
    'auto': 'auto', 'autovettura': 'auto',
    'e-bike': 'ebike', 'ebike': 'ebike', 'bici elettrica': 'ebike',
    'furgone': 'furgone', 'furg.': 'furgone'
}
mappa_tariffa = {
    'giornaliera': 'giornaliera', 'giorno': 'giornaliera', '1 giorno': 'giornaliera',
    'mezza giornata': 'mezza giornata', '1/2 giornata': 'mezza giornata', 'half day': 'mezza giornata', 'mezza': 'mezza giornata',
    'weekend': 'weekend', 'we': 'weekend', 'week-end': 'weekend',
    'settimanale': 'settimanale', 'settimana': 'settimanale', '7gg': 'settimanale',
    'settimanale+giorni': 'settimanale+giorni', 'sett.+giorni': 'settimanale+giorni',
    'settimana+giorni': 'settimanale+giorni', 'sett + gg': 'settimanale+giorni'
}
mappa_cliente = {'privato': 'privato', 'p': 'privato', 'azienda': 'azienda', 'b2b': 'azienda', 'ditta': 'azienda'}
mappa_pagamento = {
    'carta di credito': 'carta di credito', 'carta': 'carta di credito', 'cc': 'carta di credito', 'credit card': 'carta di credito',
    'carta di debito': 'carta di debito', 'carta debito': 'carta di debito', 'bancomat': 'carta di debito', 'pos': 'carta di debito',
    'bonifico': 'bonifico', 'bonifico bancario': 'bonifico', 'bon.': 'bonifico',
    'contanti': 'contanti', 'cash': 'contanti', 'contante': 'contanti',
    'satispay': 'satispay', 'paypal': 'paypal'
}
# Valori allineati a STATI_PRENOTAZIONE dell'app (buby-rent/api/app/models.py)
mappa_stato = {
    'concluso': 'conclusa', 'conclusa': 'conclusa', 'chiuso': 'conclusa', 'ok': 'conclusa', 'chiuso ok': 'conclusa',
    'annullato': 'annullata', 'annullata': 'annullata', 'cancellato': 'annullata',
    'no show': 'no_show', 'noshow': 'no_show', 'no-show': 'no_show', 'non presentato': 'no_show',
    'in corso': 'in_corso', 'in_corso': 'in_corso', 'attivo': 'in_corso', 'aperto': 'in_corso'
}

df['canale_prenotazione'] = df['canale_prenotazione'].str.strip().str.lower().replace(mappa_canale)
df['tipologia_mezzo'] = df['tipologia_mezzo'].str.strip().str.lower().replace(mappa_tipologia)
df['sede_ritiro'] = df['sede_ritiro'].str.strip().str.lower().replace(mappa_sedi).fillna(df['_sede_origine'])
df['sede_riconsegna'] = df['sede_riconsegna'].str.strip().str.lower().replace(mappa_sedi).fillna(df['sede_ritiro'])
df['tipo_tariffa'] = df['tipo_tariffa'].str.strip().str.lower().replace(mappa_tariffa)
df['tipo_cliente'] = df['tipo_cliente'].str.strip().str.lower().replace(mappa_cliente)
df['metodo_pagamento'] = df['metodo_pagamento'].str.strip().str.lower().replace(mappa_pagamento)
df['stato_noleggio'] = df['stato_noleggio'].str.strip().str.lower().replace(mappa_stato)
# L'app non ha uno stato no-show: diventa 'annullata' e l'informazione resta nella colonna no_show
df['no_show'] = np.where(df['stato_noleggio'] == 'no_show', 'si', 'no')
df['stato_noleggio'] = df['stato_noleggio'].replace({'no_show': 'annullata'})

# 4.2 Pulizia anagrafica clienti, P.IVA, email e telefoni
df['cliente_id'] = df['cliente_id'].str.strip().str.upper().replace({'-': pd.NA, 'N.D.': pd.NA})
df['nominativo_cliente'] = df['nominativo_cliente'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
df['comune_residenza'] = df['comune_residenza'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()

piva_digits = df['piva_cliente'].astype('string').str.replace(r'\D', '', regex=True)
df['piva_cliente'] = piva_digits.where(piva_digits.str.len() == 11, pd.NA)

df['email_cliente'] = (
    df['email_cliente']
    .str.strip()
    .str.lower()
    .replace({'-': pd.NA, 'n.d.': pd.NA})
    .str.replace('@exmple.', '@example.', regex=False)
    .str.replace(r'\.con$', '.com', regex=True)
    .str.replace(r'\.ti$', '.it', regex=True)
    .str.replace(r'^(info)([a-z0-9-]+\.example\.(?:it|com|net|org))$', r'\1@\2', regex=True)
    # NON inserire '@' quando precede già un punto (info@ditta.example.it)
    .str.replace(r'(?<![@.])(example\.(?:it|com|net|org))$', r'@\1', regex=True)
)

tel_digits = df['telefono_cliente'].astype('string').str.replace(r'\D', '', regex=True)
tel_digits = tel_digits.str.replace(r'^(?:0039|39)(?=\d{9,10}$)', '', regex=True)
df['telefono_cliente'] = tel_digits.where(tel_digits.str.len() >= 9, pd.NA)

# 4.2b Recupero del telefono mancante dalle altre righe dello stesso cliente
# (difetto 'cifre_mancanti' dichiarato nel README)
tel_ok = df['telefono_cliente'].notna()
tel_by_cliente = df.loc[tel_ok].groupby('cliente_id')['telefono_cliente'].first()
tel_by_email = df.loc[tel_ok].groupby('email_cliente')['telefono_cliente'].first()
df['telefono_cliente'] = (
    df['telefono_cliente']
    .fillna(df['cliente_id'].map(tel_by_cliente))
    .fillna(df['email_cliente'].map(tel_by_email))
    .astype('string')
)

# 4.3 Normalizzazione extra_selezionati, livelli carburante/batteria, danni e valutazione
df['extra_selezionati'] = (
    df['extra_selezionati']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({'-': pd.NA, 'nessuno': pd.NA})
    .str.replace(r'\s*[+;/,]\s*', '; ', regex=True)
)

mappa_livelli = {
    'pieno': '100', 'full': '100', 'p': '100',
    'tre quarti': '75', '3/4': '75',
    'metà': '50', 'meta': '50', '1/2': '50', 'mezzo': '50',
    'un quarto': '25', '1/4': '25',
    'riserva': '10', 'spia accesa': '10', 'r': '10'
}
for col_liv in ['livello_carb_batt_ritiro', 'livello_carb_batt_riconsegna']:
    s_liv = df[col_liv].astype('string').str.strip().str.lower().replace(mappa_livelli).str.replace('%', '', regex=False).str.strip()
    df[col_liv] = pd.to_numeric(s_liv, errors='coerce')

df['danni_segnalati'] = (
    df['danni_segnalati']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({'sì': 'si', 's': 'si', 'true': 'si', '1': 'si', 'n': 'no', 'false': 'no', '0': 'no'})
)

val_str = df['valutazione_cliente'].astype('string').str.strip().str.lower()
val_stars = val_str.str.count('★').replace(0, np.nan)
val_num = pd.to_numeric(val_str.str.replace('/5', '', regex=False).str.replace('stelle', '', regex=False).str.strip(), errors='coerce')
df['valutazione_cliente'] = val_stars.fillna(val_num)
df.loc[(df['valutazione_cliente'] < 1) | (df['valutazione_cliente'] > 5), 'valutazione_cliente'] = np.nan

## 5. Normalizzazione orari (`HH:MM`), parsing date e coerenza temporale

In [6]:
# 5.1 Normalizzazione orari (17h39, 12.26, 15:08:01 -> HH:MM)
for col_ora in ['ora_ritiro', 'ora_riconsegna_effettiva']:
    parts = df[col_ora].astype('string').str.strip().str.replace('[hH.]', ':', regex=True).str.extract(r'^(\d{1,2}):(\d{2})')
    df[col_ora] = np.where(parts[0].notna(), parts[0].str.zfill(2) + ':' + parts[1], pd.NA)

# 5.2 Sostituzione mesi testuali italiani e parsing con pd.to_datetime
mesi_it_num = {
    ' gen ': '/01/', ' feb ': '/02/', ' mar ': '/03/', ' apr ': '/04/',
    ' mag ': '/05/', ' giu ': '/06/', ' lug ': '/07/', ' ago ': '/08/',
    ' set ': '/09/', ' ott ': '/10/', ' nov ': '/11/', ' dic ': '/12/'
}

date_cols = ['data_prenotazione', 'data_ritiro', 'data_riconsegna_prevista', 'data_riconsegna_effettiva']
raw_durata = df['durata_giorni'].copy()

def parse(s, dayfirst):
    s = s.astype('string').str.strip().str.lower().replace({'-': pd.NA, 'n.d.': pd.NA})
    for k, v in mesi_it_num.items(): s = s.str.replace(k, v, regex=False)
    iso = s.str.match(r'^\d{4}-\d{2}-\d{2}', na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    out[iso] = pd.to_datetime(s[iso].str[:10], format='%Y-%m-%d', errors='coerce')
    o = ~iso & s.notna()
    out[o] = pd.to_datetime(s[o], format='mixed', dayfirst=dayfirst, errors='coerce')
    return out

A = pd.DataFrame({c: parse(df[c], True)  for c in date_cols})
B = pd.DataFrame({c: parse(df[c], False) for c in date_cols})

def plausibile(D):
    v = D.data_ritiro.between('2025-11-01', '2026-11-10')
    v &= D.data_riconsegna_prevista.between(D.data_ritiro, D.data_ritiro + pd.Timedelta(days=45))
    v &= D.data_prenotazione.isna() | D.data_prenotazione.between('2025-08-01', D.data_ritiro)
    v &= D.data_riconsegna_effettiva.isna() | D.data_riconsegna_effettiva.between(
         D.data_ritiro, D.data_riconsegna_prevista + pd.Timedelta(days=10))
    return v

usa_B = ~plausibile(A) & plausibile(B)
final = A.copy()
final.loc[usa_B] = B.loc[usa_B]
df[date_cols] = final

late = (df['data_riconsegna_effettiva'] - df['data_riconsegna_prevista']).dt.days > 60
df.loc[late, 'data_riconsegna_effettiva'] -= pd.DateOffset(years=1)
early = df['data_riconsegna_effettiva'] < df['data_ritiro']
df.loc[early, 'data_riconsegna_effettiva'] += pd.DateOffset(years=1)

mask_conc_na = (df['stato_noleggio'] == 'conclusa') & df['data_riconsegna_effettiva'].isna()
df.loc[mask_conc_na, 'data_riconsegna_effettiva'] = df.loc[mask_conc_na, 'data_riconsegna_prevista']

dp = (df['data_riconsegna_prevista'] - df['data_ritiro']).dt.days.clip(lower=1)
calc = np.where(df['tipo_tariffa'] == 'mezza giornata', 0.5,
        np.where(df['tipo_tariffa'] == 'weekend', 2.0, dp.astype(float)))
raw_d = pd.to_numeric(raw_durata, errors='coerce')
df['durata_giorni'] = np.where(raw_d.sub(calc).isin([0, 1]), raw_d, calc)

## 6. Pulizia variabili numeriche (chilometri e importi) e gestione outlier

In [7]:
# 6.1 Pulizia chilometraggi
for col_km in ['km_inizio', 'km_fine', 'km_percorsi']:
    s_km = (
        df[col_km]
        .astype('string')
        .str.lower()
        .str.replace('km', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(r'(?<=\d)\.(?=\d{3}$)', '', regex=True)
        .str.replace(',', '.', regex=False)
    )
    df[col_km] = pd.to_numeric(s_km, errors='coerce')

# Recupero id_veicolo e-bike mancanti tramite progressione km_fine -> km_inizio
eb_ref = df[df['id_veicolo'].notna() & (df['tipologia_mezzo'] == 'ebike')][['km_fine', 'id_veicolo']].dropna().drop_duplicates('km_fine')
sede_di = flotta.set_index('id_veicolo')['id_sede']
cand = df['km_inizio'].map(eb_ref.set_index('km_fine')['id_veicolo'])
df['id_veicolo'] = df['id_veicolo'].fillna(cand.where(cand.map(sede_di) == df['sede_ritiro']))

# Correzione cifra in più su km_fine (es. 866310 vs km_inizio 86249)
mask_km_x10 = df['km_inizio'].notna() & df['km_fine'].notna() & (df['km_fine'] > df['km_inizio'] * 5) & ((df['km_fine'] / 10) >= df['km_inizio'])
df.loc[mask_km_x10, 'km_fine'] = (df.loc[mask_km_x10, 'km_fine'] / 10).round(0)

# Correzione km_fine < km_inizio
mask_km_inv = df['km_inizio'].notna() & df['km_fine'].notna() & (df['km_fine'] < df['km_inizio'])
df.loc[mask_km_inv, 'km_fine'] = np.where(
    df.loc[mask_km_inv, 'km_percorsi'].notna() & (df.loc[mask_km_inv, 'km_percorsi'] >= 0),
    df.loc[mask_km_inv, 'km_inizio'] + df.loc[mask_km_inv, 'km_percorsi'],
    np.nan
)

# Ricalcolo coerente di km_percorsi e km_fine
df['km_percorsi'] = np.where(df['km_inizio'].notna() & df['km_fine'].notna(), df['km_fine'] - df['km_inizio'], df['km_percorsi'])
df['km_fine'] = df['km_fine'].fillna(df['km_inizio'] + df['km_percorsi'])

print("ID veicolo ancora mancanti dopo incrocio flotta + progressione km:",
      df['id_veicolo'].isna().sum())

# 6.2 Pulizia importi monetari (*_eur) e imputazione cauzione da flotta
euro_cols = ['tariffa_base_eur', 'extra_km_eur', 'extra_servizi_eur', 'penale_ritardo_eur', 'sconto_eur', 'importo_totale_eur', 'cauzione_eur']
for col_eur in euro_cols:
    s_eur = (
        df[col_eur]
        .astype('string')
        .str.replace('€', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(r'\.(?=\d{3},)', '', regex=True)
        .str.replace(',', '.', regex=False)
    )
    df[col_eur] = pd.to_numeric(s_eur, errors='coerce')

df['cauzione_eur'] = df['cauzione_eur'].fillna(df['id_veicolo'].map(flotta.set_index('id_veicolo')['cauzione_eur']))

for col_opt in ['extra_km_eur', 'extra_servizi_eur', 'penale_ritardo_eur', 'sconto_eur']:
    df[col_opt] = df[col_opt].fillna(0.0)

df['importo_totale_eur'] = df['importo_totale_eur'].abs()
mask_tb = df['tariffa_base_eur'].isna() & df['importo_totale_eur'].notna() & (df['importo_totale_eur'] > 0)
df.loc[mask_tb, 'tariffa_base_eur'] = (
    df.loc[mask_tb, 'importo_totale_eur']
    - df.loc[mask_tb, 'extra_km_eur']
    - df.loc[mask_tb, 'extra_servizi_eur']
    - df.loc[mask_tb, 'penale_ritardo_eur']
    + df.loc[mask_tb, 'sconto_eur']
).round(2)

mask_attivo = df['stato_noleggio'].isin(['conclusa', 'in_corso'])
df.loc[mask_attivo, 'importo_totale_eur'] = (
    df.loc[mask_attivo, 'tariffa_base_eur'].fillna(0.0)
    + df.loc[mask_attivo, 'extra_km_eur']
    + df.loc[mask_attivo, 'extra_servizi_eur']
    + df.loc[mask_attivo, 'penale_ritardo_eur']
    - df.loc[mask_attivo, 'sconto_eur']
).round(2)
df.loc[~mask_attivo, 'importo_totale_eur'] = df.loc[~mask_attivo, 'importo_totale_eur'].fillna(0.0)

ID veicolo ancora mancanti dopo incrocio flotta + progressione km: 4


## 7. Deduplicazione e controllo qualità post-pulizia

In [8]:
# Ordinamento per completezza della riga e rimozione duplicati sulla chiave primaria id_noleggio
df['_completezza'] = df.notna().sum(axis=1)
df_clean = (
    df.sort_values(['id_noleggio', '_completezza'], ascending=[True, False])
    .drop_duplicates(subset=['id_noleggio'], keep='first')
    .drop(columns=['_completezza'])
    .reset_index(drop=True)
)

# Validazione rispetto ai log difetti in _ground_truth
log_gt = pd.concat([pd.read_csv(f"{BASE_DATI}/_ground_truth/log_difetti_strutturati_{s}.csv") for s in SEDI], ignore_index=True)
difetti_nol = log_gt[log_gt['id_riga'].str.upper().str.startswith('NOL', na=False)]

print(f"Righe iniziali: {len(df_raw)} -> Righe pulite deduplicate: {len(df_clean)}")
print(f"Difetti iniettati nel log per i noleggi (conteggio, NON verifica di risoluzione): {len(difetti_nol)}")
assert df_clean['id_noleggio'].is_unique, "ID noleggio non univoci!"
assert (df_clean['durata_giorni'] <= 0).sum() == 0, "Presenti durate <= 0!"
assert (df_clean['data_riconsegna_prevista'] < df_clean['data_ritiro']).sum() == 0, "Riconsegna prevista < ritiro!"
assert (df_clean['data_riconsegna_effettiva'] < df_clean['data_ritiro']).sum() == 0, "Riconsegna effettiva < ritiro!"
assert (df_clean['km_percorsi'] < 0).sum() == 0, "Chilometri percorsi negativi!"
assert (df_clean['importo_totale_eur'].dropna() < 0).sum() == 0, "Presenti importi negativi!"
# Validazione dei domini canonici: intercetta regressioni delle regole di pulizia
assert set(df_clean['canale_prenotazione'].dropna()) <= {'web', 'app', 'sportello', 'telefono', 'portale partner'}
assert set(df_clean['tipologia_mezzo'].dropna()) <= {'auto', 'ebike', 'furgone'}  # TIPI_VEICOLO dell'app
assert set(df_clean['stato_noleggio'].dropna()) <= {'richiesta', 'confermata', 'in_corso', 'conclusa', 'annullata'}  # STATI_PRENOTAZIONE dell'app
assert set(df_clean['sede_ritiro'].dropna()) <= {'ALB', 'BAR', 'CAS', 'STR', 'TOR'}
assert df_clean['email_cliente'].dropna().str.match(r'^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$').all(), "Email malformate residue!"
assert df_clean['telefono_cliente'].dropna().astype(str).str.fullmatch(r'\d{9,10}').all(), "Telefoni malformati residui!"
print("Controllo qualita' superato su tutte le 5 sedi (domini canonici + email + telefoni)!")

Righe iniziali: 5143 -> Righe pulite deduplicate: 4997
Difetti iniettati nel log per i noleggi (conteggio, NON verifica di risoluzione): 20719
Controllo qualita' superato su tutte le 5 sedi (domini canonici + email + telefoni)!


## 8. Esportazione dei dataset puliti (per singola sede e consolidato)

In [9]:
cols_export = [c for c in df_clean.columns if c != '_sede_origine']

for s in SEDI:
    out_dir = f"{OUT_BASE}/{s}/strutturati"
    os.makedirs(out_dir, exist_ok=True)
    df_clean[df_clean['_sede_origine'] == s][cols_export].to_csv(f"{out_dir}/noleggi_{s}_pulito.csv", index=False)

df_clean[cols_export].to_csv(f"{OUT_BASE}/noleggi_tutti_pulito.csv", index=False)
print("Salvataggio completato in:", OUT_BASE)
df_clean[cols_export].head()

Salvataggio completato in: ../../dati_puliti


,id_noleggio,data_prenotazione,canale_prenotazione,id_veicolo,targa,tipologia_mezzo,sede_ritiro,sede_riconsegna,data_ritiro,ora_ritiro,...,penale_ritardo_eur,sconto_eur,importo_totale_eur,cauzione_eur,metodo_pagamento,stato_noleggio,danni_segnalati,valutazione_cliente,note_operatore,no_show
0,NOL-ALB-00001,2025-09-16,web,ALB-05,GR153MD,auto,ALB,ALB,2025-11-01,10:16,...,0.0,0.0,110.5,600.0,bonifico,conclusa,si,<NA>,NaN,no
1,NOL-ALB-00002,2025-09-17,sportello,ALB-01,GK284RT,auto,ALB,ALB,2025-11-01,17:39,...,0.0,0.0,393.15,300.0,carta di credito,conclusa,no,<NA>,mezzo sostituito all'ultimo,no
2,NOL-ALB-00003,2025-09-22,web,EB-ALB-003,<NA>,ebike,ALB,ALB,2025-11-06,13:36,...,0.0,0.0,13.0,100.0,carta di debito,conclusa,no,5,NaN,no
3,NOL-ALB-00004,2025-09-23,app,ALB-07,GP905TB,furgone,ALB,ALB,2025-11-07,15:15,...,0.0,0.0,93.5,600.0,carta di credito,conclusa,<NA>,4,NaN,no
4,NOL-ALB-00005,2025-09-30,web,EB-ALB-002,<NA>,ebike,ALB,ALB,2025-11-14,16:09,...,0.0,0.0,17.0,200.0,contanti,conclusa,no,<NA>,NaN,no
